In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Tue Jun  7 12:02:15 2022

@author: sunms498
"""

## Load libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from os import walk
import pickle
from pathlib import Path
from shapely.geometry import Point
import geopandas as gpd
from geopandas import GeoDataFrame
import matplotlib.colors as colors
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from pathlib import Path
from matplotlib.colors import LogNorm
import cmcrameri.cm as cmc
import cmocean as cmo
ds=2.6 #bulk density of marine soil
import sys
module_path = Path().resolve().parents[1] / "src" / "postprocessing"

# Append the directory to sys.path
sys.path.append(str(module_path))


from visualizePredictionMap import *

## Load data

In [ ]:
#Load prediction maps and uncertainty maps from the monte carlo dropout for both continental shelves and deep ocean models
filename = Path().resolve().parents[1] / "data" / "output" / "predictionmaps_TOC" / "prediction_map_TOC_supervised_men_CS_RF_v2.npy"
prediction_map_CS = np.load(filename)
prediction_map_DO = np.load(Path().resolve().parents[1] / "data" / "output" / "predictionmaps_TOC" / "prediction_map_TOC_supervised_men_DO_RF_v2.npy")



In [ ]:
Path().resolve().parents[1] / "data" / "output" / "predictionmaps_TOC" / "prediction_map_TOC_supervised_men_DO_RF_v2.npy"

In [ ]:
# Overlap both the deep ocean and the continental shelves
# pm: prediction map, um_uncertaintymap
dataset_mask = np.load(Path().resolve().parents[1] / "data" / "interim" / "masks" / "mask_deepocean.npy")
dataset_mask = np.rot90(np.rot90(np.fliplr(dataset_mask)))

# Create a copy of pm_CS
pm_overlapped = np.copy(prediction_map_DO)

# Mask the values from pm_CS where mask contains NaN values
pm_overlapped[np.isnan(dataset_mask)] = prediction_map_CS[np.isnan(dataset_mask)]



In [ ]:
#load porosity file

file_porosity = Path().resolve().parents[1] / "data" / "interim" / "porosity" / "grl53425-sup-0002-supinfo.grd"
#file_porosity = "file_porosity = "/home/sunms498/Data/paper_porosity/GPSM_Restreppo/prediction.nc""
rootgrp_porosity = xr.open_dataset(Path(file_porosity), format = 'NETCDF4')
                                                                   
dataset_porosity = rootgrp_porosity.variables['z']

dataset_numpy_porosity = dataset_porosity.to_numpy()
dataset_numpy_porosity = np.rot90(np.rot90(np.fliplr(dataset_numpy_porosity)))

In [ ]:

# make area of the cells file, based on the latitude


restrepo_features_path = Path().resolve().parents[1] / "data" / "raw" / "features" / "FeaturesPhrampusLee_TOCSedRate_updated" / "GL_LATITUDE_DD.5m.nc"
restrepo_files = []

for (dirpath, dirnames, filenames) in walk(restrepo_features_path):
    restrepo_files.extend(filenames)

restrepo_files.sort()
len(restrepo_files)

rootgrp = xr.open_dataset(restrepo_features_path, format = 'NETCDF4')
latitudes = rootgrp.variables['lat']
longitudes = rootgrp.variables['lon']


file_idx = -1
corrupted_files = 0

rootgrp = xr.open_dataset(restrepo_features_path, format = 'NETCDF4')

dataset = rootgrp.variables['data']
area = 9.297**2 * np.cos(dataset/180 * np.pi) # in km2

In [ ]:
# overlay land boundaries on the prediction and uncertainty maps

land_file = Path().resolve().parents[1] / "data" / "raw" / "island_map.npy"
land_map = np.load(land_file)
land_map[np.isnan(land_map)] = 1


pm_overlapped[land_map] = np.nan


In [ ]:
dataset_mask = np.load(Path().resolve().parents[1] / "data" / "interim" / "masks" / "mask_continentalshelves.npy")
plt.imshow(dataset_mask)

## Visualise

In [ ]:
tocGlobal = (1-(dataset_numpy_porosity/100))*ds*(pm_overlapped/100)*10
VisualiseMap(tocGlobal, cmap = cmc.tokyo, use_log_norm=True, vmin=1e-2, vmax=1, cbar_label= "TOC stock($g/cm^2$)")

In [ ]:
VisualiseMap(pm_overlapped, cmap=cmc.batlow, use_log_norm=False, vmin=0, vmax=3, cbar_label= "TOC concentration (%)")#cmc.tokyo#"cubehelix"

In [ ]:
np.save("pm_toc_Conc_after.npy",pm_overlapped)

In [ ]:
area_np = np.array(area) # in km²:  10¹⁰ cm² = 1 km²
tocGlobal_g = np.multiply(tocGlobal,area_np)*1e10 # in g

VisualiseMap(tocGlobal_g/1e15, cmap=cmc.batlow, use_log_norm=True, vmin=1e-6, vmax=1e-3, cbar_label= "TOC stock ($Pg$)")

In [ ]:
area_np = np.array(area) # in km²:  10¹⁰ cm² = 1 km²

area_np_land = area_np
area_np_land[land_map]= 0
VisualiseMap(area_np_land, cmap=cmc.oslo, vmax=90, cbar_label='Cell area ($km^2$)')

In [ ]:
tocPercent_cellwise = np.multiply(pm_overlapped,area_np) # in % km²
VisualiseMap(tocPercent_cellwise, cmap=cmo.cm.haline, use_log_norm=True,vmin=1e1, vmax=1e3, cbar_label= "TOC concentration (%)")

## Analyse

In [ ]:
# Make geo dataframes and plot them
import geopandas as gpd
from rasterio.features import geometry_mask
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


resolution = 5 / 60  # 5 arc minutes in degrees
lon = np.arange(-180, 180, resolution)
lat = np.arange(-90, 90, resolution)


tocPercent_cellwise_rot = np.rot90(np.rot90(np.fliplr(tocPercent_cellwise)))
area_np_land_rot = np.rot90(np.rot90(np.fliplr(area_np_land)))
tocGlobal_g_rot = np.rot90(np.rot90(np.fliplr(tocGlobal_g)))
continentalshelves_mask = np.load(Path().resolve().parents[1] / "data" / "interim" / "masks" / "mask_continentalshelves.npy")
#continentalshelves_mask_rot = np.rot90(np.rot90(np.fliplr(continentalshelves_mask)))

# Create a GeoDataFrame with the prediction values and their corresponding geometry
lon_mesh, lat_mesh = np.meshgrid(lon, lat)
tocPercentcCellwise_gdf = gpd.GeoDataFrame(
    data={'TOCPercent': tocPercent_cellwise_rot.flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
prediction_gdf = gpd.GeoDataFrame(
    data={'TOC': tocGlobal_g_rot.flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
area_gdf = gpd.GeoDataFrame(
    data={'area': np.array(area_np_land_rot).flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
continentalshelves_gdf = gpd.GeoDataFrame(
    data={'continentalshelves': np.array(continentalshelves_mask).flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)


# Downsample the GeoDataFrame by taking every nth point
n = 100  # Adjust this value based on your desired level of downsampling
downsampled_gdf = prediction_gdf.iloc[::n, :]

# Plot the downsampled GeoDataFrame
downsampled_gdf.plot(column='TOC', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('TOC density Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = area_gdf.iloc[::n, :]

# Plot the downsampled GeoDataFrame
downsampled_gdf.plot(column='area', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('Area Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = tocPercentcCellwise_gdf.iloc[::n, :]

downsampled_gdf.plot(column='TOCPercent', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('TOC Conc. Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = continentalshelves_gdf.iloc[::n, :]

downsampled_gdf.plot(column='continentalshelves', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('Continental shelves (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
ocean_gdf = gpd.read_file(Path().resolve().parents[1] / "data" / "interim" / "GOaS_v1_20211214" / "goas_v01.shp")
ocean_gdf.plot()
ocean_gdf

In [ ]:
# Find the sum of TOC stock, TOC Percent and area in each ocean region
joined_gdf = gpd.sjoin(prediction_gdf, ocean_gdf, predicate='within')
joined_area_gdf = gpd.sjoin(area_gdf, ocean_gdf, predicate='within')
joined_gdf_tocPercent = gpd.sjoin(tocPercentcCellwise_gdf, ocean_gdf, predicate='within')

In [ ]:
# Calculate the sum of TOC in each polygon
sum_toc_by_polygon = (joined_gdf.groupby('name')['TOC']).sum()

# Print the sum of TOC in each polygon
for ocean_name, toc_sum in sum_toc_by_polygon.items():
    print(f"Sum of TOC  in {ocean_name}: {toc_sum/1e15} Pg")

In [ ]:
# Calculate the sum of TOC Percent weighted by the area of the cells in each polygon
sum_toc_by_polygon = (joined_gdf_tocPercent.groupby('name')['TOCPercent']).sum()

# Print the sum of TOC in each polygon
for ocean_name, toc_sum in sum_toc_by_polygon.items():
    print(f"Sum of TOC Percent in {ocean_name}: {toc_sum/1e6} %km2 (x1e6)")

In [ ]:
# Calculate the area in each polygon
area_by_polygon = (joined_area_gdf.groupby('name')['area']).sum()

# Print the sum of TOC in each polygon
for ocean_name, area in area_by_polygon.items():
    print(f"Area of {ocean_name}: {area/1e6} million km^2")

In [ ]:
# Calculate the mean TOC percent for each polygon
mean_toc_percent_by_polygon = sum_toc_by_polygon / area_by_polygon

# Print the mean TOC percent in each polygon
for ocean_name, mean_toc_percent in mean_toc_percent_by_polygon.items():
    print(f"Mean TOC Percent in {ocean_name}: {mean_toc_percent} %")

In [ ]:
caspian_sea_gdf = gpd.read_file(Path().resolve().parents[1] / "data" / "interim" / "GOaS_v1_20211214" / "worldglwd1.shp")
caspian_sea_gdf.plot()
plt.title('Caspian Sea')
plt.show()
# Reproject caspian_sea_gdf to EPSG:4326
caspian_sea_gdf = caspian_sea_gdf.to_crs(epsg=4326)

# Perform spatial join
joined_gdf_caspian = gpd.sjoin(prediction_gdf, caspian_sea_gdf, how="left", predicate="within")
joined_gdf_caspian_tocPercent = gpd.sjoin(tocPercentcCellwise_gdf, caspian_sea_gdf, predicate='within')
joined_gdf_caspian_area = gpd.sjoin(area_gdf, caspian_sea_gdf, how="left", predicate="within")

# Check the TOC values inside and outside the Caspian Sea
toc_inside_caspian = np.nansum(joined_gdf_caspian[joined_gdf_caspian['lake_name'].notna()]['TOC'])
toc_outside_caspian = np.nansum(joined_gdf_caspian[joined_gdf_caspian['lake_name'].isna()]['TOC'])

print(f"TOC stock inside the Caspian Sea: {toc_inside_caspian/1e15} Pg")
#print(f"TOC stock outside the Caspian Sea: {toc_outside_caspian/1e15} Gt")

# Perform spatial join

# Check the TOC values inside and outside the Caspian Sea
area_inside_caspian = np.nansum(joined_gdf_caspian_area[joined_gdf_caspian_area['lake_name'].notna()]['area'])
area_outside_caspian = np.nansum(joined_gdf_caspian_area[joined_gdf_caspian_area['lake_name'].isna()]['area'])

print(f"Area inside the Caspian Sea: {area_inside_caspian/1e6} million km^2")
#print(f"Area outside the Caspian Sea: {area_outside_caspian/1e6} million km^2")

# Check the TOC values inside and outside the Caspian Sea
tocPercent_inside_caspian = np.nansum(joined_gdf_caspian_tocPercent[joined_gdf_caspian_tocPercent['lake_name'].notna()]['TOCPercent'])
tocPercent_outside_caspian = np.nansum(joined_gdf_caspian_tocPercent[joined_gdf_caspian_tocPercent['lake_name'].isna()]['TOCPercent'])

print(f"TOC Percent inside the Caspian Sea: {tocPercent_inside_caspian/1e6} % (1e6)")
print(f"Mean TOC Percent inside the Caspian Sea: {tocPercent_inside_caspian/area_inside_caspian} % ")



In [ ]:
# get the toc map for the continental shelves
dataset_mask = np.load(Path().resolve().parents[1] / "data" / "interim" / "masks" / "mask_deepocean.npy")
dataset_mask = np.rot90(np.rot90(np.fliplr(dataset_mask)))

# Create a copy of tocGlobal_CS
tocGlobal_g_CS_masked = np.copy(tocGlobal_g)
# Use boolean indexing to set values to nan where dataset_mask is not nan
tocGlobal_g_CS_masked[~np.isnan(dataset_mask)] = np.nan


tocPercent_cellwise_CS_masked = np.copy(tocPercent_cellwise)
# Use boolean indexing to set values to nan where dataset_mask is not nan
tocPercent_cellwise_CS_masked[~np.isnan(dataset_mask)] = np.nan

area_np_land_CS_masked = np.copy(area_np_land)
# Use boolean indexing to set values to nan where dataset_mask is not nan
area_np_land_CS_masked[~np.isnan(dataset_mask)] = np.nan

VisualiseMap(tocGlobal_g_CS_masked/1e15, cmap=cmc.batlow, use_log_norm=True, vmin=1e-6, vmax=1e-3, cbar_label= "TOC stock ($Pg$)")
VisualiseMap(tocPercent_cellwise_CS_masked, cmap=cmo.cm.haline, use_log_norm=True,vmin=1e1, vmax=1e3, cbar_label= "TOC concentration (%)")
VisualiseMap(area_np_land_CS_masked, cmap=cmc.oslo, vmax=90, cbar_label='Cell area ($km^2$)')


In [ ]:

tocPercent_cellwise_CS_masked_rot = np.rot90(np.rot90(np.fliplr(tocPercent_cellwise_CS_masked)))
area_np_land_CS_masked_rot = np.rot90(np.rot90(np.fliplr(area_np_land_CS_masked)))
tocGlobal_g_CS_masked_rot = np.rot90(np.rot90(np.fliplr(tocGlobal_g_CS_masked)))
# Create a GeoDataFrame with the prediction values and their corresponding geometry
lon_mesh, lat_mesh = np.meshgrid(lon, lat)
tocPercentcCellwise_CS_masked_gdf = gpd.GeoDataFrame(
    data={'TOCPercent': tocPercent_cellwise_CS_masked_rot.flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
prediction_CS_masked_gdf = gpd.GeoDataFrame(
    data={'TOC': tocGlobal_g_CS_masked_rot.flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
area_CS_masked_gdf = gpd.GeoDataFrame(
    data={'area': np.array(area_np_land_CS_masked_rot).flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
continentalshelves_gdf = gpd.GeoDataFrame(
    data={'continentalshelves': np.array(continentalshelves_mask).flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)


# Downsample the GeoDataFrame by taking every nth point
n = 100  # Adjust this value based on your desired level of downsampling
downsampled_gdf = prediction_CS_masked_gdf.iloc[::n, :]

# Plot the downsampled GeoDataFrame
downsampled_gdf.plot(column='TOC', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('TOC density Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = area_CS_masked_gdf.iloc[::n, :]

# Plot the downsampled GeoDataFrame
downsampled_gdf.plot(column='area', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('Area Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = tocPercentcCellwise_CS_masked_gdf.iloc[::n, :]

downsampled_gdf.plot(column='TOCPercent', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('TOC Conc. Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = continentalshelves_gdf.iloc[::n, :]

downsampled_gdf.plot(column='continentalshelves', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('Continental shelves (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# Perform a spatial join to associate each point with the corresponding polygon
joined_gdf = gpd.sjoin(prediction_CS_masked_gdf, ocean_gdf, predicate='within')
joined_area_gdf = gpd.sjoin(area_CS_masked_gdf, ocean_gdf, predicate='within')
# Perform a spatial join to associate each point with the corresponding polygon
joined_gdf_TOCPercent = gpd.sjoin(tocPercentcCellwise_CS_masked_gdf, ocean_gdf, predicate='within')

In [ ]:
# Calculate the sum of TOC in each polygon
sum_toc_by_polygon = (joined_gdf.groupby('name')['TOC']).sum()

# Print the sum of TOC in each polygon
for ocean_name, toc_sum in sum_toc_by_polygon.items():
    print(f"Sum of TOC between 0m and 200m of {ocean_name}: {toc_sum/1e15} Pg")

In [ ]:
# Calculate the sum of TOC in each polygon
sum_toc_by_polygon = (joined_gdf_TOCPercent.groupby('name')['TOCPercent']).sum()

# Print the sum of TOC in each polygon
for ocean_name, toc_sum in sum_toc_by_polygon.items():
    print(f"Sum of TOC % between 0m and 200m of {ocean_name}: {toc_sum/1e6} %km^2 (1e6)")

In [ ]:
# Calculate the sum of TOC in each polygon
area_by_polygon = (joined_area_gdf.groupby('name')['area']).sum()

# Print the sum of TOC in each polygon
for ocean_name, area in area_by_polygon.items():
    print(f"Area of  between 0m and 200m of {ocean_name}: {area/1e6} million km^2")

In [ ]:
# Calculate the mean TOC percent for each polygon
mean_toc_percent_by_polygon = sum_toc_by_polygon / area_by_polygon

# Print the mean TOC percent in each polygon
for ocean_name, mean_toc_percent in mean_toc_percent_by_polygon.items():
    print(f"Mean TOC Percent between 0m and 200m of {ocean_name}: {mean_toc_percent} %")

In [ ]:
# get the toc map for the continental shelves
dataset_mask = np.load(Path().resolve().parents[1] / "data" / "interim" / "masks" / "mask_continentalshelves.npy")
dataset_mask = np.rot90(np.rot90(np.fliplr(dataset_mask)))

# Create a copy of tocGlobal_CS
tocGlobal_g_CS_masked = np.copy(tocGlobal_g)
# Use boolean indexing to set values to nan where dataset_mask is not nan
tocGlobal_g_CS_masked[~np.isnan(dataset_mask)] = np.nan


tocPercent_cellwise_CS_masked = np.copy(tocPercent_cellwise)
# Use boolean indexing to set values to nan where dataset_mask is not nan
tocPercent_cellwise_CS_masked[~np.isnan(dataset_mask)] = np.nan

area_np_land_CS_masked = np.copy(area_np_land)
# Use boolean indexing to set values to nan where dataset_mask is not nan
area_np_land_CS_masked[~np.isnan(dataset_mask)] = np.nan

VisualiseMap(tocGlobal_g_CS_masked/1e15, cmap=cmc.batlow, use_log_norm=True, vmin=1e-6, vmax=1e-3, cbar_label= "TOC stock ($Pg$)")
VisualiseMap(tocPercent_cellwise_CS_masked, cmap=cmo.cm.haline, use_log_norm=True,vmin=1e1, vmax=1e3, cbar_label= "TOC concentration (%)")
VisualiseMap(area_np_land_CS_masked, cmap=cmc.oslo, vmax=90, cbar_label='Cell area ($km^2$)')


In [ ]:

tocPercent_cellwise_CS_masked_rot = np.rot90(np.rot90(np.fliplr(tocPercent_cellwise_CS_masked)))
area_np_land_CS_masked_rot = np.rot90(np.rot90(np.fliplr(area_np_land_CS_masked)))
tocGlobal_g_CS_masked_rot = np.rot90(np.rot90(np.fliplr(tocGlobal_g_CS_masked)))
# Create a GeoDataFrame with the prediction values and their corresponding geometry
lon_mesh, lat_mesh = np.meshgrid(lon, lat)
tocPercentcCellwise_CS_masked_gdf = gpd.GeoDataFrame(
    data={'TOCPercent': tocPercent_cellwise_CS_masked_rot.flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
prediction_CS_masked_gdf = gpd.GeoDataFrame(
    data={'TOC': tocGlobal_g_CS_masked_rot.flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
area_CS_masked_gdf = gpd.GeoDataFrame(
    data={'area': np.array(area_np_land_CS_masked_rot).flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)
continentalshelves_gdf = gpd.GeoDataFrame(
    data={'continentalshelves': np.array(continentalshelves_mask).flatten()},
    geometry=gpd.points_from_xy(lon_mesh.flatten(), lat_mesh.flatten()),
    crs='EPSG:4326'
)


# Downsample the GeoDataFrame by taking every nth point
n = 100  # Adjust this value based on your desired level of downsampling
downsampled_gdf = prediction_CS_masked_gdf.iloc[::n, :]

# Plot the downsampled GeoDataFrame
downsampled_gdf.plot(column='TOC', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('TOC density Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = area_CS_masked_gdf.iloc[::n, :]

# Plot the downsampled GeoDataFrame
downsampled_gdf.plot(column='area', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('Area Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = tocPercentcCellwise_CS_masked_gdf.iloc[::n, :]

downsampled_gdf.plot(column='TOCPercent', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('TOC Conc. Map (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

downsampled_gdf = continentalshelves_gdf.iloc[::n, :]

downsampled_gdf.plot(column='continentalshelves', cmap='viridis', legend=True, markersize=1)

# Add title and labels
plt.title('Continental shelves (Downsampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# Perform a spatial join to associate each point with the corresponding polygon
joined_gdf = gpd.sjoin(prediction_CS_masked_gdf, ocean_gdf, predicate='within')
joined_area_gdf = gpd.sjoin(area_CS_masked_gdf, ocean_gdf, predicate='within')
# Perform a spatial join to associate each point with the corresponding polygon
joined_gdf_TOCPercent = gpd.sjoin(tocPercentcCellwise_CS_masked_gdf, ocean_gdf, predicate='within')

In [ ]:
# Calculate the sum of TOC in each polygon
sum_toc_by_polygon = (joined_gdf.groupby('name')['TOC']).sum()

# Print the sum of TOC in each polygon
for ocean_name, toc_sum in sum_toc_by_polygon.items():
    print(f"Sum of TOC between 0m and 200m of {ocean_name}: {toc_sum/1e15} Pg")

In [ ]:
# Calculate the sum of TOC in each polygon
sum_toc_by_polygon = (joined_gdf_TOCPercent.groupby('name')['TOCPercent']).sum()

# Print the sum of TOC in each polygon
for ocean_name, toc_sum in sum_toc_by_polygon.items():
    print(f"Sum of TOC % between 0m and 200m of {ocean_name}: {toc_sum/1e6} %km^2 (1e6)")

In [ ]:
# Calculate the sum of TOC in each polygon
area_by_polygon = (joined_area_gdf.groupby('name')['area']).sum()

# Print the sum of TOC in each polygon
for ocean_name, area in area_by_polygon.items():
    print(f"Area of  between 0m and 200m of {ocean_name}: {area/1e6} million km^2")

In [ ]:
# Calculate the mean TOC percent for each polygon
mean_toc_percent_by_polygon = sum_toc_by_polygon / area_by_polygon

# Print the mean TOC percent in each polygon
for ocean_name, mean_toc_percent in mean_toc_percent_by_polygon.items():
    print(f"Mean TOC Percent between 0m and 200m of {ocean_name}: {mean_toc_percent} %")

In [ ]:
pm_before = np.load("pm_toc_Conc_before.npy")
pm_after = np.load("pm_toc_Conc_after.npy")
VisualiseMap(pm_after - pm_before, cmap=cmc.berlin, use_log_norm=False, vmax = 1, vmin = -1, cbar_label= "TOC concentration (%)")

In [ ]:
import numpy as np
import scipy.io as sio

# Load the NumPy array from the file
pm_after = np.load("pm_toc_Conc_after.npy")

# Save the array to a .mat file
sio.savemat("pm_toc_Conc_after.mat", {'pm_after': pm_after})

In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt

# Load the .mat file
mat_contents = sio.loadmat('pm_toc_Conc_after.mat')

# Extract the array from the loaded data
pm_after = mat_contents['pm_after']

VisualiseMap(pm_after, cmap=cmc.tokyo, use_log_norm=False, vmin=0, vmax=4, cbar_label= "TOC concentration (%)")

In [ ]:
import xarray as xr
import scipy.io as sio

# Load the .grd file
file_path = '/gxfs_work/cau/sunms498/work/nn-toc-update/data/raw/features/FeaturesPhrampusLee_TOCSedRate_updated/GL_ELEVATION_M_ASL_ETOPO2v2.5m.grd'
data = xr.open_dataset(file_path)

# Convert the relevant data to a NumPy array
# Assuming the variable of interest is 'z'
z_data = data['data'].values

# Save the array to a .mat file
sio.savemat('GL_ELEVATION_M_ASL_ETOPO2v2.mat', {'z_data': z_data})

In [ ]:
plt.imshow(data['data'])